In [1]:
# CELL 1: Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import pandas as pd
import zipfile
import os
import time
from PIL import Image
from tqdm.notebook import tqdm # Progress bar for notebooks
from utils import MetricsEngine, visual_compare

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
# CELL 2: The Data Loader & Smart Edge Sampling
def load_and_prep_data(image_path, downscale_factor=4):
    """Loads image, downscales, and splits indices into Edges vs Flat regions."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    new_w, new_h = w // downscale_factor, h // downscale_factor
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    img_np = np.array(img)
    img_norm = img_np / 255.0
    
    # Create Coordinates
    y_coords = np.linspace(-1, 1, new_h)
    x_coords = np.linspace(-1, 1, new_w)
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    
    coords = np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1)
    colors = img_norm.reshape(-1, 3)
    
    # --- UPGRADE: Smart Edge Sampling ---
    # We use Canny edge detection to find the building lines
    img_gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(img_gray, 100, 200)
    edges_flat = edges.flatten()
    
    # Separate indices
    edge_indices = np.where(edges_flat > 0)[0]
    flat_indices = np.where(edges_flat == 0)[0]
    
    return {
        "coords": torch.tensor(coords, dtype=torch.float32).to(device),
        "colors": torch.tensor(colors, dtype=torch.float32).to(device),
        "edge_indices": torch.tensor(edge_indices, dtype=torch.long).to(device),
        "flat_indices": torch.tensor(flat_indices, dtype=torch.long).to(device),
        "h": new_h, "w": new_w,
        "original_np": img_np
    }

dataset = load_and_prep_data("jcsmr-1.jpg", downscale_factor=4)
print(f"Image Resolution: {dataset['w']}x{dataset['h']} | Edges found: {len(dataset['edge_indices'])}")

Image Resolution: 1404x936 | Edges found: 86686


In [3]:
# CELL 3: The Modular Network (SIREN / ReLU)
class PositionalEncoding(nn.Module):
    def __init__(self, num_frequencies):
        super().__init__()
        self.num_frequencies = num_frequencies
        
    def forward(self, x):
        encoded = [x]
        for i in range(self.num_frequencies):
            for fn in [torch.sin, torch.cos]:
                encoded.append(fn((2.0 ** i) * torch.pi * x))
        return torch.cat(encoded, dim=-1)

class SineActivation(nn.Module):
    def forward(self, x):
        return torch.sin(x)

class ModularMLP(nn.Module):
    def __init__(self, hidden_dim, num_layers, pos_enc_freqs, activation="sine"):
        super().__init__()
        self.pos_encoder = PositionalEncoding(pos_enc_freqs)
        input_dim = 2 + 4 * pos_enc_freqs 
        
        act_layer = SineActivation() if activation.lower() == "sine" else nn.ReLU()
        
        layers = [nn.Linear(input_dim, hidden_dim), act_layer]
        for _ in range(num_layers - 2):
            layers.extend([nn.Linear(hidden_dim, hidden_dim), act_layer])
            
        layers.extend([nn.Linear(hidden_dim, 3), nn.Sigmoid()])
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(self.pos_encoder(x))


In [4]:
# CELL 4: The Training and Evaluation Engine
def train_and_evaluate(config, dataset):
    model_name = config["name"]
    print(f"\n=== Starting: {model_name} ===")
    
    model = ModularMLP(config["dim"], config["layers"], config["freqs"], config["activation"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.MSELoss()
    
    # TQDM Progress Bar
    pbar = tqdm(range(config["epochs"]), desc=model_name)
    model.train()
    
    batch_size = config["batch_size"]
    edge_ratio = config.get("edge_ratio", 0.5) # What % of batch should be edge pixels?
    
    num_edge_samples = int(batch_size * edge_ratio)
    num_flat_samples = batch_size - num_edge_samples

    for epoch in pbar:
        # Smart Sampling: Pull specific amounts from edges vs flat areas
        idx_e = torch.randint(0, len(dataset["edge_indices"]), (num_edge_samples,), device=device)
        idx_f = torch.randint(0, len(dataset["flat_indices"]), (num_flat_samples,), device=device)
        
        batch_indices = torch.cat([dataset["edge_indices"][idx_e], dataset["flat_indices"][idx_f]])
        
        batch_coords = dataset["coords"][batch_indices]
        batch_colors = dataset["colors"][batch_indices]
        
        pred = model(batch_coords)
        loss = criterion(pred, batch_colors)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch % 50 == 0:
            psnr = -10.0 * torch.log10(loss).item()
            pbar.set_postfix({"PSNR": f"{psnr:.2f}"})

    # --- EVALUATION STAGE ---
    model.eval()
    
    # 1. Measure Latency (Forward Pass)
    start_time = time.time()
    with torch.no_grad():
        predicted_colors = []
        chunk_size = 30000 # Prevent Out Of Memory
        for i in range(0, len(dataset["coords"]), chunk_size):
            chunk_coords = dataset["coords"][i:i+chunk_size]
            predicted_colors.append(model(chunk_coords))
            
        full_pred = torch.cat(predicted_colors, dim=0)
        
    if torch.cuda.is_available(): torch.cuda.synchronize()
    latency_ms = (time.time() - start_time) * 1000

    # 2. Save Output Image
    pred_np = full_pred.cpu().numpy().reshape(dataset["h"], dataset["w"], 3)
    pred_img_uint8 = (pred_np * 255).clip(0, 255).astype(np.uint8)
    out_img_path = f"images/{model_name}.png"
    cv2.imwrite(out_img_path, cv2.cvtColor(pred_img_uint8, cv2.COLOR_RGB2BGR))

    # 3. Save and Zip Model (Measure true size)
    pth_path = f"models/{model_name}.pth"
    zip_path = f"models/{model_name}.zip"
    torch.save(model.state_dict(), pth_path)
    
    # We zip it because .pth files contain unnecessary metadata
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(pth_path, arcname=f"{model_name}.pth")
    size_kb = os.path.getsize(zip_path) / 1024.0

    # 4. Compute Metrics
    engine = MetricsEngine(device)
    metrics = engine.compute_all(dataset["original_np"], pred_img_uint8)
    
    # Compile Row
    row = {"Method": model_name, "Size_KB": size_kb, "Latency_ms": latency_ms}
    row.update(metrics)
    
    return row

In [5]:
# CELL 5: The Automation Control Room
EXPERIMENTS = [
    # Baseline comparison (Proves standard Neural Networks blur images)
    {"name": "ReLU_Baseline", "activation": "relu", "layers": 5, "dim": 256, "freqs": 15, "epochs": 3000, "batch_size": 32768, "lr": 1e-3, "edge_ratio": 0.5},
    
    # The SIREN Sweep
    {"name": "SIREN_Tiny", "activation": "sine", "layers": 3, "dim": 128, "freqs": 15, "epochs": 3000, "batch_size": 32768, "lr": 1e-3, "edge_ratio": 0.5},
    {"name": "SIREN_Med",  "activation": "sine", "layers": 5, "dim": 256, "freqs": 15, "epochs": 3000, "batch_size": 32768, "lr": 1e-3, "edge_ratio": 0.5},
    {"name": "SIREN_Large","activation": "sine", "layers": 5, "dim": 512, "freqs": 15, "epochs": 3000, "batch_size": 32768, "lr": 1e-3, "edge_ratio": 0.5}
]

all_neural_results = []

for config in EXPERIMENTS:
    result = train_and_evaluate(config, dataset)
    all_neural_results.append(result)
    # Save intermediate in case of crash
    pd.DataFrame(all_neural_results).to_csv("results/neural_metrics_temp.csv", index=False)

# Final Save
df_neural = pd.DataFrame(all_neural_results)
df_neural.to_csv("results/neural_metrics.csv", index=False)
print("\nAll experiments completed and saved!")


=== Starting: ReLU_Baseline ===


ReLU_Baseline:   0%|          | 0/3000 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\vgg.pth
LPIPS loaded successfully.

=== Starting: SIREN_Tiny ===


...

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\vgg.pth
LPIPS loaded successfully.

=== Starting: SIREN_Med ===


SIREN_Med:   0%|          | 0/3000 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\vgg.pth
LPIPS loaded successfully.

=== Starting: SIREN_Large ===


SIREN_Large:   0%|          | 0/3000 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\Madhav\AppData\Local\Programs\Python\Python310\lib\site-packages\lpips\weights\v0.1\vgg.pth
LPIPS loaded successfully.

All experiments completed and saved!
